In [0]:
# Load the warehouse Delta tables

dim_customer = spark.table("workspace.default.dim_customer")
dim_date = spark.table("workspace.default.dim_date")
dim_product = spark.table("workspace.default.dim_product")
dim_payment = spark.table("workspace.default.dim_payment")
dim_status = spark.table("workspace.default.dim_status")

fact_orders = spark.table("workspace.default.fact_orders")

print("All warehouse Delta tables loaded successfully!")

All warehouse Delta tables loaded successfully!


In [0]:
print("CUSTOMER DIMENSION")
dim_customer.printSchema()

print("DATE DIMENSION")
dim_date.printSchema()

print("PRODUCT DIMENSION")
dim_product.printSchema()

print("FACT ORDERS")
fact_orders.printSchema()

CUSTOMER DIMENSION
root
 |-- customer_key: integer (nullable = true)
 |-- customer_id: integer (nullable = true)
 |-- name: string (nullable = true)
 |-- city: string (nullable = true)
 |-- segment: string (nullable = true)
 |-- signup_date: date (nullable = true)
 |-- effective_from: timestamp (nullable = true)
 |-- effective_to: timestamp (nullable = true)
 |-- is_current: boolean (nullable = true)

DATE DIMENSION
root
 |-- date: date (nullable = true)
 |-- date_key: integer (nullable = true)
 |-- day: integer (nullable = true)
 |-- month: integer (nullable = true)
 |-- month_name: string (nullable = true)
 |-- quarter: integer (nullable = true)
 |-- year: integer (nullable = true)
 |-- day_name: string (nullable = true)
 |-- week_of_year: integer (nullable = true)
 |-- is_weekend: boolean (nullable = true)

PRODUCT DIMENSION
root
 |-- product_key: integer (nullable = true)
 |-- category: string (nullable = true)

FACT ORDERS
root
 |-- order_id: integer (nullable = true)
 |-- date_ke

In [0]:
from pyspark.sql.functions import (
    col,
    sum,
    count,
    avg,
    max,
    countDistinct,
    round,
    year,
    month
)

# Join orders with date dimension
orders_enriched = (
    fact_orders
    .join(dim_date, on="date_key", how="left")
    .join(dim_customer, on="customer_key", how="left")
    .join(dim_product, on="product_key", how="left")
    .join(dim_payment, on="payment_key", how="left")
    .join(dim_status, on="status_key", how="left")
)

print("Fact table successfully joined with all dimensions!")

display(orders_enriched.limit(10))

Fact table successfully joined with all dimensions!


status_key,payment_key,product_key,customer_key,date_key,order_id,quantity,amount,source,date,day,month,month_name,quarter,year,day_name,week_of_year,is_weekend,customer_id,name,city,segment,signup_date,effective_from,effective_to,is_current,category,payment_method,status
2,2,6,583,20260924,166585,3,80021.99,web,2026-09-24,24,9,September,3,2026,Thursday,39,false,567,Mahnoor Khan,Hyderabad,Consumer,2023-09-29,2023-09-29T00:00:00.000Z,2026-12-31T00:00:00.000Z,true,Home,card,delivered
2,4,3,1267,20260928,188311,2,32242.92,web,2026-09-28,28,9,September,3,2026,Monday,40,false,1231,Imran Butt,Rawalpindi,Corporate,2024-04-18,2024-04-18T00:00:00.000Z,2026-12-31T00:00:00.000Z,true,Electronics,wallet,delivered
2,3,2,252,20260925,218039,1,1057.99,web,2026-09-25,25,9,September,3,2026,Friday,39,false,245,Maryam Chaudhry,Islamabad,Consumer,2022-11-23,2022-11-23T00:00:00.000Z,2026-12-31T00:00:00.000Z,true,Books,cash_on_delivery,delivered
4,2,1,1669,20260909,261988,5,9547.4,web,2026-09-09,9,9,September,3,2026,Wednesday,37,false,1626,Amna Mirza,Lahore,Consumer,2022-08-16,2022-08-16T00:00:00.000Z,2026-12-31T00:00:00.000Z,true,Beauty,card,returned
3,4,2,985,20260930,168034,2,6786.9,web,2026-09-30,30,9,September,3,2026,Wednesday,40,false,956,Usman Siddiqui,Rawalpindi,Consumer,2023-10-24,2023-10-24T00:00:00.000Z,2026-12-31T00:00:00.000Z,true,Books,wallet,pending
2,2,1,583,20260924,102491,1,1321.8,web,2026-09-24,24,9,September,3,2026,Thursday,39,false,567,Mahnoor Khan,Hyderabad,Consumer,2023-09-29,2023-09-29T00:00:00.000Z,2026-12-31T00:00:00.000Z,true,Beauty,card,delivered
2,3,6,556,20260925,246213,1,16378.53,web,2026-09-25,25,9,September,3,2026,Friday,39,false,540,Fatima Raza,Karachi,Consumer,2023-06-09,2023-06-09T00:00:00.000Z,2026-12-31T00:00:00.000Z,true,Home,cash_on_delivery,delivered
5,4,1,663,20260929,158189,2,14700.06,web,2026-09-29,29,9,September,3,2026,Tuesday,40,false,642,Hamza Mirza,Lahore,Small Business,2023-11-24,2023-11-24T00:00:00.000Z,2026-12-31T00:00:00.000Z,true,Beauty,wallet,shipped
5,4,6,1316,20260930,277556,1,8243.39,web,2026-09-30,30,9,September,3,2026,Wednesday,40,false,1279,Imran Sheikh,Karachi,Consumer,2023-02-14,2023-02-14T00:00:00.000Z,2026-12-31T00:00:00.000Z,true,Home,wallet,shipped
5,1,2,287,20260928,248000,1,517.29,web,2026-09-28,28,9,September,3,2026,Monday,40,false,280,Ayesha Malik,Islamabad,Consumer,2022-10-28,2022-10-28T00:00:00.000Z,2026-12-31T00:00:00.000Z,true,Books,bank_transfer,shipped


Gold table 1 – Monthly Sales Summary

- How much revenue did the shop generate each month?
- How many orders were placed?
- How many units were sold?
- What was the average order value?

In [0]:
from pyspark.sql.functions import (
    sum as spark_sum,
    countDistinct,
    avg,
    round,
    col
)

# Create monthly sales summary
gold_sales_summary = (
    orders_enriched
    .groupBy("year", "month", "month_name")
    .agg(
        countDistinct("order_id").alias("total_orders"),
        spark_sum("amount").alias("total_revenue"),
        spark_sum("quantity").alias("units_sold"),
        round(avg("amount"), 2).alias("average_order_value")
    )
    .orderBy("year", "month")
)

# Save as a Gold Delta table
gold_sales_summary.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.default.gold_sales_summary")

print("Gold sales summary created successfully!")

display(gold_sales_summary)

Gold sales summary created successfully!


year,month,month_name,total_orders,total_revenue,units_sold,average_order_value
2025,1,January,9390,2.2143142273000014E8,15727,23581.62
2025,2,February,8405,1.9600994773000014E8,14092,23320.64
2025,3,March,9337,2.3022961463000005E8,15851,24657.77
2025,4,April,8843,2.1804362102000022E8,14920,24657.2
2025,5,May,9282,2.1521769191000012E8,15557,23186.56
2025,6,June,8974,2.1679049310999984E8,15301,24157.62
2025,7,July,9237,2.185375018899998E8,15650,23658.93
2025,8,August,9514,2.2746835701E8,16019,23908.8
2025,9,September,9038,2.089478134699999E8,15452,23118.81
2025,10,October,9319,2.2319604989000005E8,15683,23950.64


Gold table 2 – Customer Summary


This table helps the business understand customer purchasing behaviour.
- Which customers spend the most?
- How many orders does each customer place?
- Which customer segments generate revenue?
- When did each customer last order?

In [0]:
from pyspark.sql.functions import max as spark_max

gold_customer_summary = (
    orders_enriched
    .groupBy(
        "customer_id",
        "name",
        "city",
        "segment"
    )
    .agg(
        countDistinct("order_id").alias("total_orders"),
        spark_sum("amount").alias("total_spent"),
        spark_sum("quantity").alias("total_units"),
        round(avg("amount"), 2).alias("average_order_value"),
        spark_max("date").alias("last_order_date")
    )
    .orderBy(col("total_spent").desc())
)

# Save as a Gold Delta table
gold_customer_summary.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.default.gold_customer_summary")

print("Gold customer summary created successfully!")

display(gold_customer_summary.limit(20))

Gold customer summary created successfully!


customer_id,name,city,segment,total_orders,total_spent,total_units,average_order_value,last_order_date
111,Saad Khan,Islamabad,Small Business,617,1.652945193E7,1033,26790.04,2026-10-24
596,Iqra Qureshi,Faisalabad,Consumer,592,1.611325338E7,971,27218.33,2026-10-24
537,Imran Chaudhry,Karachi,Consumer,597,1.5677610579999998E7,1022,26260.65,2026-10-24
410,Mahnoor Qureshi,Karachi,Corporate,564,1.5651096920000004E7,950,27750.17,2026-10-24
989,Hira Chaudhry,Faisalabad,Small Business,580,1.5491334369999994E7,990,26709.2,2026-10-24
1200,Omar Raza,Faisalabad,Consumer,637,1.5277915340000002E7,1012,23984.17,2026-10-24
312,Ali Butt,Lahore,Corporate,616,1.5162398890000004E7,1048,24614.28,2026-10-24
929,Maryam Malik,Karachi,Consumer,571,1.5100758219999995E7,947,26446.16,2026-10-24
1572,Hassan Qureshi,Lahore,Consumer,570,1.5030853959999995E7,1004,26369.92,2026-10-23
461,Saad Qureshi,Faisalabad,Consumer,608,1.4997449699999997E7,996,24666.86,2026-10-22


Gold table 3 – Product Performance

This table helps the shop identify its best-performing product categories.



In [0]:
gold_product_performance = (
    orders_enriched
    .groupBy("category")
    .agg(
        countDistinct("order_id").alias("total_orders"),
        spark_sum("quantity").alias("total_units_sold"),
        spark_sum("amount").alias("total_revenue"),
        round(avg("amount"), 2).alias("average_order_value")
    )
    .orderBy(col("total_revenue").desc())
)

# Save as a Gold Delta table
gold_product_performance.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.default.gold_product_performance")

print("Gold product performance created successfully!")

display(gold_product_performance)

Gold product performance created successfully!


category,total_orders,total_units_sold,total_revenue,average_order_value
Electronics,33397,56538,2.619554330980008E9,78436.82
Home,33477,56686,1.176875055989997E9,35154.73
Fashion,33098,55767,4.4364625680999964E8,13404.02
Beauty,33358,56251,2.3578821279999924E8,7068.42
Grocery,33623,56742,1.7923074568999985E8,5330.6
Books,33036,56088,1.0683442999000022E8,3233.88


In [0]:
gold_tables = [
    "gold_sales_summary",
    "gold_customer_summary",
    "gold_product_performance"
]

for table_name in gold_tables:
    df = spark.table(f"workspace.default.{table_name}")

    print(f"\n{table_name}")
    print("Records:", df.count())
    df.printSchema()


gold_sales_summary
Records: 22
root
 |-- year: integer (nullable = true)
 |-- month: integer (nullable = true)
 |-- month_name: string (nullable = true)
 |-- total_orders: long (nullable = true)
 |-- total_revenue: double (nullable = true)
 |-- units_sold: long (nullable = true)
 |-- average_order_value: double (nullable = true)


gold_customer_summary
Records: 2057
root
 |-- customer_id: integer (nullable = true)
 |-- name: string (nullable = true)
 |-- city: string (nullable = true)
 |-- segment: string (nullable = true)
 |-- total_orders: long (nullable = true)
 |-- total_spent: double (nullable = true)
 |-- total_units: long (nullable = true)
 |-- average_order_value: double (nullable = true)
 |-- last_order_date: date (nullable = true)


gold_product_performance
Records: 6
root
 |-- category: string (nullable = true)
 |-- total_orders: long (nullable = true)
 |-- total_units_sold: long (nullable = true)
 |-- total_revenue: double (nullable = true)
 |-- average_order_value: double

In [0]:
from pyspark.sql.functions import col, countDistinct

# Get orders placed during the churn window
orders_in_churn_window = (
    orders_enriched
    .filter(
        (col("date") >= "2026-08-01") &
        (col("date") <= "2026-10-24")
    )
)

# Count unique customers who placed orders
active_customers = orders_in_churn_window.select(
    "customer_id"
).distinct().count()

print("Customers who ordered during churn window:", active_customers)
print("Customers who did not order:", 2000 - active_customers)

Customers who ordered during churn window: 1998
Customers who did not order: 2
